# Step 4 — MuJoCo Wipe-Table Evaluation

Full visual benchmark: dataset oracle + ESN + **proximity-gated cloth grasp** (blended attach/release, no teleport) + **wipe task metrics** + video.

Step 3 is **dual-process only** (VLA + ESN latency integration). This notebook replays
**one episode** of `G1_Dex1_Wipe_Table` (ep.~0, ~12 s) with the same tokens/proprio the ESN was trained on.
Do not loop by default — multi-loop 60 s videos can exceed hundreds of MB.

```bash
MUJOCO_GL=egl python3 -m src.step4_mujoco_evaluation --episode 0
```


In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
RESEARCH_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RESEARCH_DIR = RESEARCH_DIR.resolve()
os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step4_mujoco_evaluation'}")

Research root : /home/aihimekpen/research_summer_2026/research
Results go to : /home/aihimekpen/research_summer_2026/research/results/step4_mujoco_evaluation


In [ ]:
INIT_EPISODE = 0
DURATION_S = None              # None = exactly one episode (~12 s); avoid multi-loop GB videos
LOOP_EPISODE = False           # set True only with an explicit duration_s > episode length
CONTROL_MODE = "kinematic"     # kinematic | pd
CONTROL_HZ = 100.0
VLA_HZ = 2.0
RECORD_VIDEO = True
VIDEO_FPS = 30.0               # 30 fps × ~12 s ≈ 360-frame output
DEVICE = "cuda"
MJCF_PATH = None
ESN_CHECKPOINT = None

_dur = "1 episode" if DURATION_S is None else f"{DURATION_S:.0f}s"
print(f"episode={INIT_EPISODE} | duration={_dur} | loop={LOOP_EPISODE} | video @ {VIDEO_FPS:.0f} fps")


In [3]:
import json
import torch

from src.paths import results_path
from src.step3_dual_thread_mujoco import resolve_esn_checkpoint, resolve_mjcf_path
from src.step4_mujoco_evaluation import (
    MuJoCoEvalConfig,
    MuJoCoWipeEvaluator,
    print_eval_summary,
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required.")

mjcf = resolve_mjcf_path(MJCF_PATH)
ckpt = resolve_esn_checkpoint(ESN_CHECKPOINT)
out_dir = results_path("step4_mujoco_evaluation")
video_path = out_dir / "table_wipe_benchmark.mp4"

config = MuJoCoEvalConfig(
    mjcf_path=mjcf,
    esn_checkpoint=str(ckpt),
    init_episode=INIT_EPISODE,
    duration_s=DURATION_S,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    control_mode=CONTROL_MODE,
    record_video=RECORD_VIDEO,
    video_path=video_path,
    video_fps=VIDEO_FPS,
    device=DEVICE,
    loop_episode=LOOP_EPISODE,
)

stats = MuJoCoWipeEvaluator(config).run()

report_path = out_dir / "mujoco_eval_report.json"
report = {
    "init_episode": INIT_EPISODE,
    "control_mode": CONTROL_MODE,
    "tracking_rmse": stats.tracking_rmse,
    "grasp_frames": stats.grasp_frames,
    "trajectory_steps": stats.trajectory_steps,
    "video_path": stats.video_path,
    "task_metrics": stats.task_metrics.to_dict() if stats.task_metrics else None,
}
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print_eval_summary(stats, report_path=report_path)

if stats.task_metrics is not None:
    tm = stats.task_metrics
    print(f"\nBenchmark: max_cloth_jump={tm.max_cloth_jump_m:.4f}m | "
          f"wipe_path={tm.wipe_path_length_m:.3f}m | "
          f"grasp_ok={tm.grasp_success}")

2026-07-30 15:54:23,688 [INFO] Loading episode 0 from unitreerobotics/G1_Dex1_Wipe_Table @ 100 Hz (VLA hold 2.0 Hz)
2026-07-30 15:54:57,675 [INFO] Episode 0: 1231 steps (12.31s). Target 60.0s → 6000 sim steps (5 loops).
2026-07-30 15:54:59,403 [INFO] Loaded ESN checkpoint from /home/aihimekpen/research_summer_2026/research/models/esn_cuda_ridge/esn_cuda_ridge_best.pth (v2)
2026-07-30 15:54:59,404 [INFO]   MSE=0.000001 | jerk=0.000005 | step_hz=2885.6689104063307
2026-07-30 15:54:59,470 [INFO] ESN reservoir warmed up on 50 dataset ticks (washout=50).
2026-07-30 15:55:11,837 [INFO] ESN reservoir warmed up on 50 dataset ticks (washout=50).
2026-07-30 15:55:24,187 [INFO] ESN reservoir warmed up on 50 dataset ticks (washout=50).
2026-07-30 15:55:36,536 [INFO] ESN reservoir warmed up on 50 dataset ticks (washout=50).
2026-07-30 15:55:48,885 [INFO] ESN reservoir warmed up on 50 dataset ticks (washout=50).
2026-07-30 15:55:59,687 [WARNING] imageio H.264 export unavailable: No module named 'ima


  Step 4 — MuJoCo Wipe-Table Evaluation (Dataset Oracle)
  Episode           : 0
  Control mode      : kinematic
  Steps             : 6,000  (5 episode loop(s))
  Tracking RMSE     : 0.00137 rad  (MSE=1.88e-06)
  Cloth grasped     : 1047/6000 frames
  --- Wipe task benchmarks ---
  Max cloth jump    : 0.0385 m  (mean 0.0007 m)
  Grasp proximity   : 0.1396 m  (success=True)
  False attach      : 8 frames
  Wipe path (XY)    : 3.460 m
  Table contact     : 0.4% of grasp frames
  Mean step latency : 6.299 ms  (158.7 Hz)
  Report JSON       : /home/aihimekpen/research_summer_2026/research/results/step4_mujoco_evaluation/mujoco_eval_report.json
  Benchmark video   : /home/aihimekpen/research_summer_2026/research/results/step4_mujoco_evaluation/table_wipe_benchmark.mp4

Benchmark: max_cloth_jump=0.0385m | wipe_path=3.460m | grasp_ok=True


In [4]:
from IPython.display import Video, display

if stats.video_path:
    display(Video(stats.video_path, embed=True, width=640))
else:
    print("No video recorded.")